# ThinkOrQuit: High-Throughput Reasoning Trace Collector (Google Colab)

This notebook allows running inference with open-weight reasoning models (e.g., `Qwen/Qwen2.5-7B-Instruct`, `DeepSeek-R1-Distill-Qwen-7B`) on a free Google Colab T4 GPU to sample multi-step Chain-of-Thought paths, extract candidate answers, and compute semantic entropy and martingale drift at intermediate checkpoints.

In [ ]:
# 1. Install dependencies
!pip install -q torch transformers datasets accelerate bitsandbytes

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "Qwen/Qwen2.5-7B-Instruct"
print(f"Loading model: {model_id} on GPU...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    device_map="auto"
)
print("Model loaded successfully!")

In [ ]:
# Multi-path sampling at checkpoints
def sample_reasoning_paths(prompt, num_paths=5, max_new_tokens=256, temperature=0.7):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        num_return_sequences=num_paths,
        do_sample=True,
        temperature=temperature,
        top_p=0.95,
        pad_token_id=tokenizer.eos_token_id
    )
    input_len = inputs["input_ids"].shape[1]
    return [tokenizer.decode(out[input_len:], skip_special_tokens=True) for out in outputs]